# 💹 Notebook 06 — Order Execution Optimizer & Slippage Simulation
> **Purpose:** Use CNN-LSTM direction signals to build an ML-directed order slicer.
> Simulate execution of a 10,000-share order and compare slippage vs TWAP and VWAP.

**Inputs:** `data/lobster_features.parquet`, `data/cnn_lstm_model.pt`, `data/scaler.joblib`  
**Outputs:** `data/execution_results.csv`, slippage comparison plots

---

## 6.1  Load model, scaler, and clean data

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
sns.set_theme(style='darkgrid')
%matplotlib inline

from utils.feature_builder import FEATURE_COLS_42, build_sequences, apply_scaler
from utils.slippage_simulator import (
    twap_execute, vwap_execute, ml_directed_execute, compare_strategies
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load data
df = pd.read_parquet('data/lobster_features.parquet')
scaler = joblib.load('data/scaler.joblib')

print(f'Loaded {len(df):,} rows')

In [ ]:
# Rebuild CNN-LSTM model and load weights
class CNNLSTM(torch.nn.Module):
    def __init__(self, input_size=42, cnn_channels=[64,128,256],
                 lstm_hidden=128, lstm_layers=2, n_classes=3, dropout=0.3):
        super().__init__()
        cnn_layers = []
        in_ch = input_size
        for out_ch in cnn_channels:
            cnn_layers += [
                torch.nn.Conv1d(in_ch, out_ch, 3, padding=1),
                torch.nn.BatchNorm1d(out_ch), torch.nn.ReLU(),
                torch.nn.Dropout(dropout/2)
            ]
            in_ch = out_ch
        self.cnn  = torch.nn.Sequential(*cnn_layers)
        self.lstm = torch.nn.LSTM(cnn_channels[-1], lstm_hidden, lstm_layers,
                                   batch_first=True, bidirectional=True,
                                   dropout=dropout if lstm_layers>1 else 0)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(lstm_hidden*2, 128), torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(128, 64), torch.nn.ReLU(),
            torch.nn.Dropout(dropout), torch.nn.Linear(64, n_classes)
        )
    def forward(self, x):
        x = self.cnn(x.permute(0,2,1)).permute(0,2,1)
        out, _ = self.lstm(x)
        return self.head(out[:,-1,:])

model = CNNLSTM().to(DEVICE)
model.load_state_dict(torch.load('data/cnn_lstm_model.pt', map_location=DEVICE))
model.eval()
print('CNN-LSTM model loaded.')

## 6.2  Generate signals for full dataset

In [ ]:
from utils.feature_builder import build_feature_matrix

X_df  = build_feature_matrix(df, FEATURE_COLS_42)
X_raw = X_df.values

# Scale
X_scaled = scaler.transform(X_raw).astype(np.float32)

# Build sequences
SEQ_LEN = 20
N = len(X_scaled)
X_all_seq = np.stack([X_scaled[i:i+SEQ_LEN] for i in range(N-SEQ_LEN)]).astype(np.float32)

# Batch inference
BATCH = 512
all_signals = []
with torch.no_grad():
    for i in range(0, len(X_all_seq), BATCH):
        Xb = torch.tensor(X_all_seq[i:i+BATCH]).to(DEVICE)
        preds = model(Xb).argmax(1).cpu().numpy() - 1  # remap back to {-1,0,+1}
        all_signals.extend(preds)

# Pad first SEQ_LEN rows with 0 (neutral)
signals_full = np.array([0]*SEQ_LEN + all_signals)
df_sim = df.iloc[:len(signals_full)].copy()
df_sim['signal'] = signals_full
print(f'Signals generated for {len(df_sim):,} events')
print(f'Signal dist: DOWN={np.sum(signals_full==-1):,}  FLAT={np.sum(signals_full==0):,}  UP={np.sum(signals_full==1):,}')

## 6.3  Simulate order execution — 10,000 shares

In [ ]:
# Use the middle portion of the day (avoid open/close)
sim_start = len(df_sim) // 4
sim_end   = 3 * len(df_sim) // 4
df_exec   = df_sim.iloc[sim_start:sim_end].reset_index(drop=True)
signals   = df_exec['signal'].values

TOTAL_SHARES = 10_000
N_SLICES     = 10

# ── Run all three strategies ──
result_twap = twap_execute(df_exec, TOTAL_SHARES, N_SLICES, side='buy')
result_vwap = vwap_execute(df_exec, TOTAL_SHARES, N_SLICES, side='buy')
result_ml   = ml_directed_execute(df_exec, signals, TOTAL_SHARES,
                                   N_SLICES, side='buy', aggression=2.0)

cmp = compare_strategies(result_twap, result_vwap, result_ml, reference='TWAP')
display(cmp.style.background_gradient(cmap='RdYlGn', axis=0))

## 6.4  Slippage comparison plot

In [ ]:
strategies  = ['TWAP', 'VWAP', 'ML-Directed']
slippages   = [result_twap.slippage_bps, result_vwap.slippage_bps, result_ml.slippage_bps]
improvements = [0,
                result_twap.slippage_bps - result_vwap.slippage_bps,
                result_twap.slippage_bps - result_ml.slippage_bps]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['steelblue', 'darkorange', 'seagreen']
bars = axes[0].bar(strategies, slippages, color=colors, width=0.5)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_ylabel('Slippage (bps)')
axes[0].set_title('Execution Slippage by Strategy')
for bar, v in zip(bars, slippages):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.01,
                 f'{v:.2f} bps', ha='center', va='bottom', fontsize=9)

bars2 = axes[1].bar(strategies, improvements, color=colors, width=0.5)
axes[1].set_ylabel('Slippage saved vs TWAP (bps)')
axes[1].set_title('Improvement vs TWAP Baseline')
for bar, v in zip(bars2, improvements):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.005,
                 f'+{v:.2f}' if v >= 0 else f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle(f'Order Execution Simulation — {TOTAL_SHARES:,} Shares', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_slippage_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 6.5  Fill price timeline

In [ ]:
def plot_fills(result, ax, color, label):
    idxs   = [f['idx'] for f in result.fills]
    prices = [f['avg_price'] for f in result.fills]
    sizes  = [f['shares'] for f in result.fills]
    ax.scatter(idxs, prices, s=[s/50 for s in sizes], c=color, alpha=0.7, label=label)
    ax.plot(idxs, prices, c=color, alpha=0.4, lw=1)

fig, ax = plt.subplots(figsize=(13, 5))
# Background mid price
ax.plot(df_exec.index, df_exec['mid_price'], color='gray', lw=0.5,
        alpha=0.5, label='Mid price')

plot_fills(result_twap, ax, 'steelblue',  'TWAP fills')
plot_fills(result_vwap, ax, 'darkorange', 'VWAP fills')
plot_fills(result_ml,   ax, 'seagreen',   'ML-Directed fills')

ax.set_xlabel('Event index')
ax.set_ylabel('Fill price ($)')
ax.set_title('Fill Prices vs Mid Price — All Strategies')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_fill_timeline.png', dpi=120, bbox_inches='tight')
plt.show()

## 6.6  Multi-window Monte Carlo simulation

In [ ]:
"""Run the simulation across 50 random windows to estimate average slippage improvement."""
import random
random.seed(42)
np.random.seed(42)

WINDOW_SIZE = 5000
N_RUNS      = 50
results_mc  = []

for run in range(N_RUNS):
    start = random.randint(0, len(df_sim) - WINDOW_SIZE - 1)
    df_w  = df_sim.iloc[start:start+WINDOW_SIZE].reset_index(drop=True)
    sigs  = df_w['signal'].values

    r_twap = twap_execute(df_w, TOTAL_SHARES, N_SLICES)
    r_vwap = vwap_execute(df_w, TOTAL_SHARES, N_SLICES)
    r_ml   = ml_directed_execute(df_w, sigs, TOTAL_SHARES, N_SLICES)

    results_mc.append({
        'run':        run,
        'twap_slip':  r_twap.slippage_bps,
        'vwap_slip':  r_vwap.slippage_bps,
        'ml_slip':    r_ml.slippage_bps,
        'ml_vs_twap': r_twap.slippage_bps - r_ml.slippage_bps
    })

mc_df = pd.DataFrame(results_mc)
print(mc_df[['twap_slip','vwap_slip','ml_slip','ml_vs_twap']].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(mc_df['twap_slip'], bins=20, alpha=0.6, label='TWAP', color='steelblue')
axes[0].hist(mc_df['vwap_slip'], bins=20, alpha=0.6, label='VWAP', color='darkorange')
axes[0].hist(mc_df['ml_slip'],   bins=20, alpha=0.6, label='ML',   color='seagreen')
axes[0].set_title('Slippage Distribution (50 windows)')
axes[0].set_xlabel('Slippage (bps)'); axes[0].legend()

axes[1].hist(mc_df['ml_vs_twap'], bins=20, color='seagreen', edgecolor='black', alpha=0.8)
axes[1].axvline(mc_df['ml_vs_twap'].mean(), color='red', ls='--',
                label=f'Mean: {mc_df["ml_vs_twap"].mean():.2f} bps')
axes[1].set_title('ML improvement over TWAP (bps)')
axes[1].set_xlabel('Saved bps'); axes[1].legend()

plt.suptitle('Monte Carlo Execution Simulation (50 runs)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/fig_monte_carlo.png', dpi=120, bbox_inches='tight')
plt.show()

mc_df.to_csv('data/execution_results.csv', index=False)
print(f'\nMean slippage improvement vs TWAP: {mc_df["ml_vs_twap"].mean():.2f} bps')
print(f'Equivalent to {mc_df["ml_vs_twap"].mean()/mc_df["twap_slip"].mean()*100:.1f}% reduction')

---
> ✅ **Execution simulation complete.** Proceed to `07_streamlit_dashboard.py` to launch the live dashboard.